## imports + config

In [40]:
import os
import glob
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

DATA_DIR = r"data/chase/raw"
BANK_CODE = "Chase"

DB_CONFIG = dict(
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432",
    database="postgres",
)


## helpers

In [ ]:
type_map = {
    "Sale": "purchase",
    "Payment": "payment",
    "Fee": "fee",
    "Refund": "refund",
    "Return": "refund",
    "Adjustment": "other"
}

def get_account_last4_from_filename(filename):
    # Expect filenames like Chase1664_ActivityYYYYMMDD_YYYYMMDD_YYYYMMDD.CSV
    base = os.path.basename(filename)
    # Extract the 4 digits after "Chase"
    # Example: Chase1664_Activity...
    start = base.find("Chase")
    if start == -1:
        raise ValueError(f"Filename does not contain 'Chase': {base}")
    last4 = base[start+5:start+9]
    if not last4.isdigit():
        raise ValueError(f"Could not parse last4 from filename: {base}")
    return last4

def get_account_id(cur, bank_code, last4):
    cur.execute("""
        SELECT a.account_id
        FROM accounts a
        JOIN banks b ON a.bank_id = b.bank_id
        WHERE b.bank_code = %s AND a.account_last4 = %s
    """, (bank_code, last4))
    row = cur.fetchone()
    if not row:
        raise ValueError(f"No account_id for bank={bank_code} last4={last4}")
    return row[0]

def build_canonical(df, account_id, bank_code, last4, source_filename):
    canonical = pd.DataFrame()
    canonical["transaction_date"] = pd.to_datetime(df["Transaction Date"]).dt.date
    canonical["account_id"] = account_id  # Assign AFTER transaction_date exists
    canonical["posted_date"] = pd.to_datetime(df["Post Date"], errors="coerce").dt.date
    canonical["amount_cents"] = (df["Amount"].astype(float) * 100).round().astype('int64')
    canonical["description"] = df["Description"].fillna("").astype(str)
    canonical["raw_type"] = df["Type"].fillna("").astype(str)
    canonical["memo"] = df.get("Memo", pd.Series([""] * len(df))).fillna("").astype(str)

    canonical["transaction_type"] = canonical["raw_type"].map(type_map).fillna("other")

    canonical["source_filename"] = source_filename
    canonical["source_row_id"] = df.index + 1

    normalized_desc = canonical["description"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
    canonical["possible_duplicate_key"] = (
        bank_code + "|" + last4 + "|" +
        canonical["transaction_date"].astype(str) + "|" +
        canonical["amount_cents"].astype(str) + "|" +
        normalized_desc
    )

    canonical["category_id"] = None
    canonical["merchant_id"] = None
    canonical["address"] = None
    canonical["reference_number"] = None
    canonical["raw_metadata"] = None

    return canonical


## connect

In [42]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()


## ingest loop

In [43]:
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.CSV")))

sql = """
INSERT INTO transactions (
  account_id, transaction_date, amount_cents, description, transaction_type,
  source_filename, source_row_id, possible_duplicate_key,
  posted_date, category_id, merchant_id, raw_type, memo, address, reference_number, raw_metadata
) VALUES %s
"""

success = 0
failed = []

for path in files:
    try:
        last4 = get_account_last4_from_filename(path)
        account_id = get_account_id(cur, BANK_CODE, last4)

        df = pd.read_csv(path)
        canonical = build_canonical(df, account_id, BANK_CODE, last4, os.path.basename(path))

        rows = canonical[[
            "account_id","transaction_date","amount_cents","description","transaction_type",
            "source_filename","source_row_id","possible_duplicate_key",
            "posted_date","category_id","merchant_id","raw_type","memo","address","reference_number","raw_metadata"
        ]].values.tolist()

        execute_values(cur, sql, rows)
        conn.commit()
        success += 1
        print(f"OK: {os.path.basename(path)} ({len(rows)} rows)")
    except Exception as e:
        conn.rollback()
        failed.append((path, str(e)))
        print(f"FAIL: {os.path.basename(path)} -> {e}")

print(f"\nDone. Success: {success}, Failed: {len(failed)}")


FAIL: Chase1664_Activity20221126_20221231_20241127.CSV -> can't adapt type 'NAType'
FAIL: Chase1664_Activity20230101_20231231_20241127.CSV -> can't adapt type 'NAType'
FAIL: Chase1664_Activity20240101_20241231_20241127.CSV -> can't adapt type 'NAType'
FAIL: Chase1664_Activity20250101_20251231_20260105.CSV -> can't adapt type 'NAType'
FAIL: Chase9197_Activity20221126_20221231_20241127.CSV -> can't adapt type 'NAType'
FAIL: Chase9197_Activity20230101_20231231_20241127.CSV -> can't adapt type 'NAType'
FAIL: Chase9197_Activity20240101_20241231_20241127.CSV -> can't adapt type 'NAType'
FAIL: Chase9197_Activity20250101_20251231_20260105.CSV -> can't adapt type 'NAType'

Done. Success: 0, Failed: 8


## verify

In [44]:
cur.execute("SELECT COUNT(*) FROM transactions;")
cur.fetchone()


(0,)

## close

In [45]:
cur.close()
conn.close()


In [46]:
 # Add this right after creating amount_cents
print(canonical["amount_cents"].describe())
print(f"Max: {canonical['amount_cents'].max()}")
print(f"Min: {canonical['amount_cents'].min()}")
print(f"Any NaN: {canonical['amount_cents'].isna().any()}")
print(f"Any inf: {(canonical['amount_cents'] == float('inf')).any()}")

count    6.920000e+02
mean     3.954480e+01
std      1.817518e+05
min     -1.849400e+06
25%     -2.898250e+03
50%     -1.299000e+03
75%     -4.712500e+02
max      3.357436e+06
Name: amount_cents, dtype: float64
Max: 3357436
Min: -1849400
Any NaN: False
Any inf: False


In [47]:
  # After creating canonical["amount_cents"], add these checks:
print(f"Dtype: {canonical['amount_cents'].dtype}")
print(f"Sample values: {canonical['amount_cents'].head().tolist()}")

  # Also check what's being sent to the database:
rows = canonical[[
    "account_id","transaction_date","amount_cents","description","transaction_type",
    "source_filename","source_row_id","possible_duplicate_key",
    "posted_date","category_id","merchant_id","raw_type","memo","address","reference_number","raw_metadata"
]].values.tolist()

print(f"First row types: {[type(x) for x in rows[0]]}")
print(f"First row amount_cents: {rows[0][2]} (type: {type(rows[0][2])})")

Dtype: int64
Sample values: [-22995, -2135, -1980, -3000, -5550]
First row types: [<class 'int'>, <class 'datetime.date'>, <class 'int'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'int'>, <class 'str'>, <class 'datetime.date'>, <class 'pandas._libs.missing.NAType'>, <class 'pandas._libs.missing.NAType'>, <class 'str'>, <class 'str'>, <class 'NoneType'>, <class 'NoneType'>, <class 'NoneType'>]
First row amount_cents: -22995 (type: <class 'int'>)
